In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("IPL.csv")

C:\Users\Dwitiyash\AppData\Local\Temp\ipykernel_22636\4237585619.py:1: DtypeWarning: Columns (0: review_batter, 1: team_reviewed, 2: review_decision, 3: umpire, 4: season, 5: superover_winner, 6: result_type, 7: method, 8: event_match_no) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("IPL.csv")


In [3]:
df_5yr = df[df["year"].between(2021, 2025)].copy()

In [4]:
innings_scores = (
    df_5yr.groupby(
        ["batter", "match_id", "innings"],
        as_index=False
    )
    .agg(
        runs=("runs_batter", "sum")
    )
)

innings_scores.head()

,batter,match_id,innings,runs
0,A Badoni,1304050,1,54
1,A Badoni,1304053,2,19
2,A Badoni,1304058,1,19
3,A Badoni,1304061,2,10
4,A Badoni,1304066,2,5


In [5]:
highest_score = (
    innings_scores.groupby("batter")["runs"]
    .max()
    .reset_index(name="highest_score")
)

In [6]:
thirties = (
    innings_scores[innings_scores["runs"] >= 30]
    .groupby("batter")
    .size()
    .reset_index(name="thirties")
)

In [7]:
fifties = (
    innings_scores[
        (innings_scores["runs"] >= 50) &
        (innings_scores["runs"] < 100)
    ]
    .groupby("batter")
    .size()
    .reset_index(name="fifties")
)

In [8]:
hundreds = (
    innings_scores[
        innings_scores["runs"] >= 100
    ]
    .groupby("batter")
    .size()
    .reset_index(name="hundreds")
)

In [9]:
dismissals = (
    df_5yr[
        (df_5yr["player_out"].notna()) &
        (df_5yr["striker_out"] == 1)
    ][["player_out", "match_id", "innings"]]
    .drop_duplicates()
    .rename(columns={"player_out": "batter"})
)

In [10]:
duck_df = innings_scores.merge(
    dismissals,
    on=["batter", "match_id", "innings"],
    how="inner"
)

In [11]:
ducks = (
    duck_df[duck_df["runs"] == 0]
    .groupby("batter")
    .size()
    .reset_index(name="ducks")
)

In [12]:
milestones = highest_score.copy()

milestones = milestones.merge(thirties, on="batter", how="left")
milestones = milestones.merge(fifties, on="batter", how="left")
milestones = milestones.merge(hundreds, on="batter", how="left")
milestones = milestones.merge(ducks, on="batter", how="left")

In [13]:
milestones = milestones.fillna(0)

milestones[["thirties","fifties","hundreds","ducks"]] = (
    milestones[["thirties","fifties","hundreds","ducks"]]
    .astype(int)
)

In [14]:
milestones.isnull().sum()

batter           0
highest_score    0
thirties         0
fifties          0
hundreds         0
ducks            0
dtype: int64

In [15]:
milestones["batter"].duplicated().sum()

np.int64(0)

In [16]:
milestones.sort_values(
    "highest_score",
    ascending=False
).head(10)

,batter,highest_score,thirties,fifties,hundreds,ducks
25,Abhishek Sharma,141,27,9,1,3
219,Q de Kock,140,15,10,1,1
289,Shubman Gill,129,44,19,4,4
327,YBK Jaiswal,124,31,15,2,3
171,MP Stoinis,124,12,5,1,5
103,JC Buttler,124,33,13,7,7
275,SV Samson,119,30,13,1,3
243,RR Pant,118,20,7,1,2
172,MR Marsh,117,14,9,1,6
311,V Kohli,113,42,24,3,4


In [17]:
milestones.sort_values(
    "hundreds",
    ascending=False
).head(10)

,batter,highest_score,thirties,fifties,hundreds,ducks
103,JC Buttler,124,33,13,7,7
289,Shubman Gill,129,44,19,4,4
128,KL Rahul,112,34,19,3,2
311,V Kohli,113,42,24,3,4
43,B Sai Sudharsan,108,29,12,2,0
85,H Klaasen,105,21,7,2,0
234,RD Gaikwad,108,35,17,2,5
258,SA Yadav,103,35,18,2,5
327,YBK Jaiswal,124,31,15,2,3
314,V Suryavanshi,101,4,1,1,1


In [18]:
milestones.sort_values(
    "fifties",
    ascending=False
).head(10)

,batter,highest_score,thirties,fifties,hundreds,ducks
311,V Kohli,113,42,24,3,4
77,F du Plessis,96,34,23,0,4
289,Shubman Gill,129,44,19,4,4
128,KL Rahul,112,34,19,3,2
258,SA Yadav,103,35,18,2,5
234,RD Gaikwad,108,35,17,2,5
327,YBK Jaiswal,124,31,15,2,3
62,DA Warner,92,20,14,0,4
275,SV Samson,119,30,13,1,3
103,JC Buttler,124,33,13,7,7


In [19]:
milestones.sort_values(
    "ducks",
    ascending=False
).head(10)

,batter,highest_score,thirties,fifties,hundreds,ducks
270,SP Narine,109,9,3,1,10
249,Rashid Khan,79,5,1,0,10
83,GJ Maxwell,78,19,12,0,9
190,N Pooran,87,29,12,0,7
103,JC Buttler,124,33,13,7,7
123,KD Karthik,83,11,3,0,7
233,RD Chahar,25,0,0,0,6
172,MR Marsh,117,14,9,1,6
58,D Padikkal,101,18,6,1,6
216,PWH de Silva,18,0,0,0,5


In [20]:
milestones.sort_values(
    "ducks",
    ascending=False
).head(10)

,batter,highest_score,thirties,fifties,hundreds,ducks
270,SP Narine,109,9,3,1,10
249,Rashid Khan,79,5,1,0,10
83,GJ Maxwell,78,19,12,0,9
190,N Pooran,87,29,12,0,7
103,JC Buttler,124,33,13,7,7
123,KD Karthik,83,11,3,0,7
233,RD Chahar,25,0,0,0,6
172,MR Marsh,117,14,9,1,6
58,D Padikkal,101,18,6,1,6
216,PWH de Silva,18,0,0,0,5


In [21]:
milestones.to_csv(
    "milestone_stats.csv",
    index=False
)

print("Milestone Analytics dataset exported successfully!")

Milestone Analytics dataset exported successfully!
